### Precision, Recall, mAP확인하기

In [1]:
!pip install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.8 MB/s eta 0:00:00


In [2]:
from ultralytics import YOLO

model = YOLO('yolo11n.pt')
metrics = model.val(
    data = r'/content/drive/MyDrive/aquarium_dataset/aquarium_dataset/data.yaml',
    imgsz=640
)
print('Precision :', metrics.box.mp)
print('Recall :', metrics.box.mr)
print('mAP50 :', metrics.box.map50)
print('mAP50-95 :', metrics.box.map)
# Precision : 0.02657698596149982
# Recall : 0.0926869970987618
# mAP50 : 0.01988449241985285       #IOU 0.5기준에서 계산한 AP (1에 가까워야 좋아.)
# mAP50-95 : 0.011751987471109789   # mAP50 보다 좀더 엄격한 기준
# 점수가 낮은 이유는 백본은 coco dataset(label 80개)
# aquarium_dataset(label 7개 : fish, jellyfish, penguins, sharks, puffins, stingrays, starfish)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 12.0±24.6 ms, read: 1.2±2.3 MB/s, size: 61.3 KB)
val: Scanning /content/drive/MyDrive/aquarium_dataset/aquarium_dataset/valid/labels... 127 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 127/127 2.7it/s 46.3s
val: New cache created: /content/drive/MyDrive/aquarium_dataset/aquarium_dataset/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 5.5s/it 44.1s
                   all        127        909     0.0266     0.0927     0.0199     0.0118
                person         63        459     0.0481      0.218       0.02    0.00962
               bicycle          9        155          0          0          0          0
                   car         17        104      0.026     0.0673     0.0018   0.000596
     

In [ ]:
# 카메라가 있는 장치에서 사용하기 - 종료 : q
# YOLO 웹캠 객체 감지 프로그램 ------------------------
import cv2
import time
import os
from ultralytics import YOLO

# 설정값
MODEL_PATH = "yolo11n.pt"
WINDOW_NAME = "YOLO Webcam Detection"
SAVE_DIR = "saved"

# 감지할 객체
TARGET_LABELS = {
    "cell phone", "laptop", "keyboard", "mouse", "cup", "book", "backpack",
    "handbag", "keyboard", "umbrella", "toothbrush"
}

CONFIDENCE_THRESHOLD = 0.55   # 최소 신뢰도
SAVE_COOLDOWN = 5  # 저장 간격(초)
FRAME_DELAY = 0.03  # 프레임 처리 간격

# 웹캠 해상도
FRAME_WIDTH = 640
FRAME_HEIGHT = 480

# 저장 폴더 준비
os.makedirs(SAVE_DIR, exist_ok=True)

print("YOLO 모델 로드 중...")
model = YOLO(MODEL_PATH)
print("모델 로드 완료")

# 웹캠 열기
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("웹캠을 열 수 없습니다.")
    exit()

cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_HEIGHT)
print("웹캠 연결 성공")

# 창 생성
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, 900, 700)

# 객체 저장 시간 기록
last_saved = {}

# 색상 정의
BOX_COLOR = (0, 255, 0)
TEXT_COLOR = (255, 255, 255)

# FPS 계산용
prev_time = time.time()

# 메인 루프
while True:
    success, frame = cap.read()

    if not success:
        print("프레임 읽기 실패")
        break

    # 좌우 반전
    frame = cv2.flip(frame, 1)

    # YOLO 추론
    results = model(
        frame,
        verbose=False
    )

    # 객체 처리
    for result in results:
        boxes = result.boxes
        for box in boxes:
            class_id = int(box.cls[0])
            label = model.names[class_id]
            confidence = float(box.conf[0])
            # 신뢰도 필터
            if confidence < CONFIDENCE_THRESHOLD:
                continue

            # 원하는 객체만
            if label not in TARGET_LABELS:
                continue

            # 좌표
            x1, y1, x2, y2 = map(int, box.xyxy[0])

            # 박스 그리기
            cv2.rectangle(
                frame, (x1, y1), (x2, y2), BOX_COLOR, 2
            )

            text = f"{label} {confidence:.2f}"

            cv2.putText(
                frame, text, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, TEXT_COLOR, 2
            )

            # 저장 처리
            current_time = time.time()
            last_time = last_saved.get(label, 0)

            if current_time - last_time > SAVE_COOLDOWN:
                filename = (
                    f"{label}_{int(current_time)}.jpg"
                )

                filepath = os.path.join(SAVE_DIR, filename)
                cv2.imwrite(filepath, frame)
                print(f"[저장 완료] {filepath}")
                last_saved[label] = current_time

    # FPS 계산
    current = time.time()
    fps = 1 / (current - prev_time)
    prev_time = current

    cv2.putText(
        frame, f"FPS: {fps:.1f}", (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2
    )

    # 화면 출력
    cv2.imshow(WINDOW_NAME, frame)

    # 종료 키
    key = cv2.waitKey(1) & 0xFF
    if key == ord("q") or key == 27:
        break

    # CPU 사용량 감소
    time.sleep(FRAME_DELAY)

# 종료 처리
cap.release()
cv2.destroyAllWindows()
print("프로그램 종료")